In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = """You are a travel agent. 
IMPORTANT: When using the search-flight tool, you MUST use the following rules:
1. Dates must be in 'dd/mm/yyyy' format (e.g., 31/03/2026).
2. All dates must be in the future.
3. If the user asks for a one-way flight, ensure you still provide the correct parameters required by the tool.
4. Use only English Language
No follow up questions."""

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Create an effective prompt template for getting flights")]},
    config
    )

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Create an effective prompt template for getting flights', additional_kwargs={}, response_metadata={}, id='853471ad-5418-442a-bbbd-9cf9c2d02641'),
              AIMessage(content='Here\'s an effective prompt template for flight searches:\n\n**Flight Search Request Template**\n\n1. **Departure Location**: [City/airport name or code]  \n   *Example: "Paris CDG" or "London"*\n\n2. **Destination**: [City/airport name or code]  \n   *Example: "New York JFK" or "Tokyo"*\n\n3. **Travel Dates**:  \n   - Departure: [dd/mm/yyyy]  \n     *Example: "25/12/2023"*  \n   - Return (if round-trip): [dd/mm/yyyy]  \n     *Example: "05/01/2024"*\n\n4. **Preferences**:  \n   - Cabin Class: [M (Economy), W (Premium), C (Business), F (First)]  \n     *Default: M if unspecified*  \n   - Currency: [ISO code]  \n     *Example: EUR, USD, GBP*  \n   - Sort By: [price, duration, quality, or date]  \n     *Default: date if unspecified*  \n   - Date Flexibility: [0-3 days]  \n     

In [7]:
print(response["messages"][-1].content)

Here's an effective prompt template for flight searches:

**Flight Search Request Template**

1. **Departure Location**: [City/airport name or code]  
   *Example: "Paris CDG" or "London"*

2. **Destination**: [City/airport name or code]  
   *Example: "New York JFK" or "Tokyo"*

3. **Travel Dates**:  
   - Departure: [dd/mm/yyyy]  
     *Example: "25/12/2023"*  
   - Return (if round-trip): [dd/mm/yyyy]  
     *Example: "05/01/2024"*

4. **Preferences**:  
   - Cabin Class: [M (Economy), W (Premium), C (Business), F (First)]  
     *Default: M if unspecified*  
   - Currency: [ISO code]  
     *Example: EUR, USD, GBP*  
   - Sort By: [price, duration, quality, or date]  
     *Default: date if unspecified*  
   - Date Flexibility: [0-3 days]  
     *Example: "2" for ±2 days around selected dates*

**Example Request**:  
"Find flights from Paris CDG to Tokyo Haneda for departure on 25/12/2023, return on 05/01/2024. Sort by price, economy class, EUR currency, and allow 1-day flexibility

In [8]:
user_prompt = """Find flights from New York to London for departure on 
                04/04/2026, return on 02/05/2026. Sort by price, 
                economy class, USD currency, and allow 1-day flexibility."""

In [9]:
from langchain.messages import HumanMessage

response = await agent.ainvoke(
    {"messages": [HumanMessage(content=user_prompt)]},
    config
    )

pprint(response)

{'messages': [HumanMessage(content='Create an effective prompt template for getting flights', additional_kwargs={}, response_metadata={}, id='853471ad-5418-442a-bbbd-9cf9c2d02641'),
              AIMessage(content='Here\'s an effective prompt template for flight searches:\n\n**Flight Search Request Template**\n\n1. **Departure Location**: [City/airport name or code]  \n   *Example: "Paris CDG" or "London"*\n\n2. **Destination**: [City/airport name or code]  \n   *Example: "New York JFK" or "Tokyo"*\n\n3. **Travel Dates**:  \n   - Departure: [dd/mm/yyyy]  \n     *Example: "25/12/2023"*  \n   - Return (if round-trip): [dd/mm/yyyy]  \n     *Example: "05/01/2024"*\n\n4. **Preferences**:  \n   - Cabin Class: [M (Economy), W (Premium), C (Business), F (First)]  \n     *Default: M if unspecified*  \n   - Currency: [ISO code]  \n     *Example: EUR, USD, GBP*  \n   - Sort By: [price, duration, quality, or date]  \n     *Default: date if unspecified*  \n   - Date Flexibility: [0-3 days]  \n     

In [ ]:
user_prompt = """Find flights from New York to London for departure on 
                04/04/2026, return on 02/05/2026. Sort by price, 
                economy class, USD currency, and allow 1-day flexibility."""

In [ ]:
print(response["messages"][-1].content)